In [0]:
%run ../01_Bronze/00_setup

In [0]:
# update the robot status if their shelves are now empty

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Determine Robot Status based on Shelf occupancy
# (Using the same robot_status_check logic from the previous step)

#robot_status_check = (all_shelf_statuses.groupBy("robot_id")
robot_status_check = (shelf_updates.groupBy("robot_id")
    .agg(F.collect_set("status").alias("status_set"))
    .withColumn("calculated_status", 
        F.when((F.size(F.col("status_set")) == 1) & (F.array_contains(F.col("status_set"), "idle")), "Idle")
        .otherwise("Picking")))

# 2. Apply Battery Drain, Low-Battery Logic, and Final Status
robot_updates_df = (robot_status_check
    .join(spark.table(f"{catalog_name}.{silver_database_name}.robots"), "robot_id")
    # Get the most recent state for each robot
    .withColumn("row_num", F.row_number().over(Window.partitionBy("robot_id").orderBy(F.col("Updated_Timestamp").desc())))
    .filter("row_num = 1")
    .withColumn("new_battery_level", F.greatest(F.lit(0), F.col("current_battery_pct") - 2))
    .withColumn("status", 
        F.when(F.col("new_battery_level") < 10, "Charging")
        .otherwise(F.col("calculated_status")))
    .select(
        "robot_id",
        "status",
        "max_weight_capacity",
        F.col("new_battery_level").alias("battery_level"),
        F.current_timestamp().alias("event_timestamp"),
        F.lit(current_run_uuid).alias("_batch_id"),
        F.current_timestamp().alias("_ingest_timestamp")
    ))

# 3. Filter for available (Idle) or maintaining (Charging) robots
final_robot_output_df = robot_updates_df.filter(F.col("status").isin("Idle", "Charging"))

display(final_robot_output_df)

(final_robot_output_df
 .drop("_metadata")
 .write.format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{source_volume_path}/robot_events"))